# Math for Data Science — Combined Course Notebook (Beginner-Friendly)

**This single notebook brings together:**
- Derivatives, gradients & optimization basics
- Probability concepts for modeling uncertainty
- Covariance, correlation & PCA for feature understanding
- Math concepts directly applied to ML algorithms

Each section is written in very simple language, includes formulas, short examples, runnable code, plots, and exercises.

---
## Table of Contents
1. Derivatives, Gradients & Optimization
2. Probability & Uncertainty
3. Covariance, Correlation & PCA
4. Math Applied to ML Algorithms
5. Exercises & Solutions

---
Run the code cells in order. If a cell requires `scipy` or `sklearn` they are used in optional sections.

## Setup — imports and helper functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (7,4)

from IPython.display import display

def describe_array(arr):
    arr = np.asarray(arr)
    out = {
        'count': int(arr.size),
        'mean': float(arr.mean()),
        'median': float(np.median(arr)),
        'mode': Counter(arr).most_common(1)[0][0] if arr.size>0 else None,
        'var(ddof=1)': float(arr.var(ddof=1)) if arr.size>1 else float('nan'),
        'std(ddof=1)': float(arr.std(ddof=1)) if arr.size>1 else float('nan'),
        'min': float(arr.min()) if arr.size>0 else None,
        'max': float(arr.max()) if arr.size>0 else None
    }
    df = __import__('pandas').DataFrame(list(out.items()), columns=['stat', 'value'])
    display(df)

print('Setup ready')

# 1 — Derivatives, Gradients & Optimization (Simple)

This section explains **how things change** and how models learn by following the slope downhill.


## 1.1 Derivative — idea & formula
**Idea:** derivative tells how fast a function changes.

**Formula:**

$$f'(x) = \lim_{h\to0} \frac{f(x+h)-f(x)}{h}$$

**Simple example:** $f(x)=x^2$ → $f'(x)=2x$.


In [ ]:
def f(x):
    return x**2

def numerical_derivative(func, x, h=1e-6):
    return (func(x+h)-func(x-h))/(2*h)

for x in [-2, -1, 0, 1, 2]:
    print(f'x={x}, f(x)={f(x)}, f\'(x)≈{numerical_derivative(f,x):.4f}')

## 1.2 Gradient — multivariable derivative
For a function of many variables, gradient is a vector of partial derivatives.
Example: $f(x,y)=x^2+y^2$ → $\nabla f=[2x,2y]$.


In [ ]:
def grad_f(x,y):
    return np.array([2*x, 2*y])
print('grad at (1,2)=', grad_f(1,2))

## 1.3 Gradient Descent — how models learn (intuitive)
**Update rule:**

$$\theta \leftarrow \theta - \alpha \nabla J(\theta)$$

We use this to minimize loss (error). α is the learning rate (step size). Smaller α = slower but safer; larger α = faster but may overshoot.


In [ ]:
# Toy example: linear regression with gradient descent
np.random.seed(0)
X = 2 * np.random.rand(100)
true_w0, true_w1 = 1.5, 2.7
Y = true_w0 + true_w1 * X + np.random.randn(100) * 0.5
X_b = np.c_[np.ones((100,1)), X]
theta = np.random.randn(2)
alpha = 0.1
m = len(Y)
history = []
for i in range(200):
    gradients = (2/m) * X_b.T.dot(X_b.dot(theta) - Y)
    theta = theta - alpha * gradients
    history.append(np.mean((X_b.dot(theta)-Y)**2))
print('estimated theta:', theta)
plt.plot(history)
plt.title('MSE during gradient descent')
plt.xlabel('iteration')
plt.ylabel('MSE')
plt.show()

### 1.4 Quick exercises (try them)
- Change `alpha` and see what happens to convergence.
- Try fewer iterations or different starting `theta`.


# 2 — Probability Concepts for Modeling Uncertainty

Probability helps us reason when outcomes are not certain.


## 2.1 Probability & Expected Value
**Expected value** (average outcome):

$$E[X] = \sum x_i P(x_i)$$

Example: biased coin with P(heads)=0.7, X=1 for heads → E[X]=0.7.


In [ ]:
p = 0.7
E = 1*p + 0*(1-p)
print('Expected value:', E)
# simulate
np.random.seed(1)
sim = np.random.choice([0,1], size=1000, p=[1-p,p])
print('Simulated mean (approx):', sim.mean())

## 2.2 Variance — how much outcomes vary
**Formula:**

$$Var(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$$


In [ ]:
vals = np.array([0,1])
probs = np.array([0.3,0.7])
E = np.sum(vals*probs)
Var = np.sum((vals-E)**2 * probs)
E, Var

## 2.3 Common distributions — quick look
- Normal (bell curve) — many measurements approx normal.
- Binomial — count of successes in fixed trials.
- Poisson — count of rare events.


In [ ]:
np.random.seed(2)
plt.hist(np.random.normal(0,1,1000), bins=30); plt.title('Normal(0,1)'); plt.show()
plt.hist(np.random.binomial(10,0.3,1000), bins=range(0,12)); plt.title('Binomial(10,0.3)'); plt.show()
plt.hist(np.random.poisson(3,1000), bins=range(0,12)); plt.title('Poisson(3)'); plt.show()

### 2.4 Simple Bayesian intuition (very short)
Use prior belief + data to form an updated belief (posterior). Example: coin fairness updated after flips. (Optional: requires `scipy` for Beta functions.)

### 2.5 Exercises
- Simulate 1000 flips of a biased coin and compute sample mean & variance.
- Sample Normal(μ=5, σ=2) 1000 times and estimate P(X>7).


# 3 — Covariance, Correlation & PCA (Feature Understanding)

These tools help us find relationships between variables and reduce dimensions.

## 3.1 Covariance — how two variables move together
**Formula:**

$$cov(X,Y)=\frac{1}{n-1}\sum_{i=1}^n (x_i-\bar{x})(y_i-\bar{y})$$


In [ ]:
np.random.seed(2)
x = np.random.randn(200)
y = 0.8*x + np.random.randn(200)*0.4
print('covariance matrix:\n', np.cov(x,y))
plt.scatter(x,y, alpha=0.5); plt.title('Scatter: correlated variables'); plt.show()

## 3.2 Correlation — scaled covariance
**Formula:**

$$\rho_{X,Y} = \frac{cov(X,Y)}{\sigma_X \sigma_Y}$$

Range is [-1, 1].

In [ ]:
print('correlation coefficient:', np.corrcoef(x,y)[0,1])

## 3.3 PCA — intuition & steps
PCA finds the main directions (principal components) that capture most variance. Steps:
1. Center the data
2. Compute SVD (or covariance & eigendecomposition)
3. Project onto top components


In [ ]:
X = np.vstack([x,y]).T
Xc = X - X.mean(axis=0)
U,s,VT = np.linalg.svd(Xc, full_matrices=False)
pc1 = VT[0]
proj = Xc.dot(pc1)
print('first PC direction:', pc1)
plt.scatter(X[:,0], X[:,1], alpha=0.4)
mean = X.mean(axis=0)
line = np.vstack([-3*pc1,3*pc1]) + mean
plt.plot(line[:,0], line[:,1], color='red'); plt.title('Data and PC1'); plt.show()
explained = (s**2)/(len(Xc)-1)
print('explained variance per PC:', explained)
print('proportion explained by PC1:', explained[0]/explained.sum())

### 3.4 Exercises
- Compute covariance & correlation for two synthetic variables.
- Run PCA on a 3D dataset and report variance explained by the first component.


# 4 — Math Concepts Directly Applied to ML Algorithms

Short demos showing math use in common ML tools: linear regression, logistic regression, PCA in pipelines.

## 4.1 Linear Regression — closed form & gradient descent
Closed-form (normal equation):

$$\theta = (X^T X)^{-1} X^T y$$

We'll compare closed-form to gradient descent result.

In [ ]:
# create data
np.random.seed(0)
X = 2 * np.random.rand(100,1)
y = 4 + 3*X[:,0] + np.random.randn(100)*0.5

# closed-form (normal equation)
X_b = np.c_[np.ones((100,1)), X]
theta_best = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
print('theta (normal eq):', theta_best)

# solve with simple gradient descent
theta = np.random.randn(2)
alpha = 0.1
m = len(y)
for i in range(200):
    gradients = (2/m) * X_b.T.dot(X_b.dot(theta) - y)
    theta = theta - alpha * gradients
print('theta (GD):', theta)

## 4.2 Logistic Regression — idea & sigmoid
Logistic regression maps linear score to probability using the sigmoid function:

$$\sigma(z)=\frac{1}{1+e^{-z}}$$

Training maximizes likelihood (often via gradient-based methods).

In [ ]:
import math
def sigmoid(z):
    return 1/(1+math.exp(-z))
print('sigmoid(0)=', sigmoid(0), 'sigmoid(2)=', sigmoid(2))

## 4.3 PCA in ML pipelines
Use PCA to reduce features before training models — speeds up training and can reduce noise.

### 4.4 Exercises
- Fit linear regression with gradient descent and compare coefficients with closed-form.
- Implement one gradient step of logistic regression for a single sample (compute gradient of log-likelihood).

# 5 — Exercises & Solutions (try then run solutions)

### Exercise A: Derivative & Gradient
Compute derivative of f(x)=x^2 at x=3 and gradient of f(x,y)=x^2+y^2 at (1,2).

In [ ]:
print('derivative approx f\'(3)=', (f(3+1e-6)-f(3-1e-6))/(2e-6))
print('gradient at (1,2)=', grad_f(1,2))

### Exercise B: Probability
Simulate 1000 biased coin flips with p=0.65. Compute sample mean and variance.

In [ ]:
np.random.seed(10)
sim = np.random.choice([0,1], size=1000, p=[0.35,0.65])
print('sample mean', sim.mean())
print('sample var', sim.var(ddof=1))

### Exercise C: Covariance, Correlation & PCA
Create two correlated variables and run PCA; show proportion of variance explained by first PC.

In [ ]:
np.random.seed(11)
x1 = np.random.randn(300)
x2 = 0.9*x1 + np.random.randn(300)*0.2
X3 = np.vstack([x1,x2]).T
X3c = X3 - X3.mean(axis=0)
U,s,VT = np.linalg.svd(X3c, full_matrices=False)
expl = (s**2)/(len(X3c)-1)
print('prop explained by PC1:', expl[0]/expl.sum())

### Exercise D: ML application
Fit linear regression using normal equation and verify predictions on a few samples.

In [ ]:
# reuse data from section 4
X_sample = np.array([[1, 0.5],[1,1.5],[1,2.0]])
print('predictions (closed-form theta):', X_sample.dot(theta_best))

## Final notes for instructors
- Encourage students to change random seeds, noise levels and parameters to see effects.
- Use plots to build intuition: histograms, scatter, PC line.
- For large datasets, discuss numerical stability and why closed-form inverses are not used.

---

### End of Combined Course Notebook
Feel free to ask for a slide deck, printable PDF, quizzes, or interactive widgets for classroom demos.